In [ ]:
import codecs
from math import sqrt

In [24]:
class Recommender:

    def __init__(self, data, k=1, metric='pearson', n=5):
        """ initialize recommender
        currently, if data is dictionary the recommender is initialized
        to it.
        For all other data types of data, no initialization occurs
        k is the k value for k nearest neighbor
        metric is which distance formula to use
        n is the maximum number of recommendations to make"""
        self.k = k
        self.n = n
        self.username2id = {}
        self.userid2name = {}
        self.productid2name = {}

        # The following two variables are used for Slope One
        self.frequencies = {}
        self.deviations = {}    

    def convertProductID2name(self, id):
        """Given product id number return product name"""
        if id in self.productid2name:
            return self.productid2name[id]
        else:
            return id

    def loadMovieDB(self, path=''):
        """loads the Movie dataset. Path is where the dataset are
        located"""
        self.data = {}

        #first load movie ratings into u.data

        f = codecs.open(path + "u.data", 'r', 'utf8')
        for line in f:
            #separate line into fields
            fields = line.split('\t')
            if fields[0] in self.data:
                self.data[fields[0]][fields[1]] = int(fields[2])
            else:
                self.data[fields[0]] = {fields[1] : int(fields[2])}
        f.close()

    def computeDeviations(self):
        # for each person in the data:
        # get their ratings
        for ratings in self.data.values():
            #for each item & rating in that set of ratings:
            for (item, rating) in ratings.items():
                self.frequencies.setdefault(item, {})
                self.deviations.setdefault(item, {})
                # for each item2 & rating2 in that set of ratings
                for (item2, rating2) in ratings.items():
                    if item != item2:
                        # add the difference between the ratings to our computation
                        self.frequencies[item].setdefault(item2, 0)
                        self.deviations[item].setdefault(item2,0.0)
                        self.frequencies[item][item2] += 1
                        self.deviations[item][item2] += rating- rating2

        for (item, ratings) in self.deviations.items():
            for item2 in ratings:
                ratings[item2] /= self.frequencies[item][item2]     

    def slopeOneRecommendations(self, userRatings):
        recommendations = {}
        frequencies = {}
        # for every item and rating in the user's recommendations
        for (userItem, userRating) in userRatings.items():
            # for every item in our dataset that the user didn't rate
            for (diffItem, diffRatings) in self.deviations.items():
                if diffItem not in userRatings and userItem in self.deviations[diffItem]:
                    freq = self.frequencies[diffItem][userItem]
                    recommendations.setdefault(diffItem, 0.0)
                    frequencies.setdefault(diffItem, 0)
                    # add to the running sum representing the numerator of the formula
                    recommendations[diffItem] += (diffRatings[userItem] + userRating) * freq
                    # keep a running sum of the frequency of diffitem
                    frequencies[diffItem] += freq
        recommendations = [(self.convertProductID2name(k),v / frequencies[k]) for (k, v) in recommendations.items()]
        # finally sort and return
        recommendations.sort(key=lambda artistTuple: artistTuple[1], reverse = True)
        return recommendations 

    def adjustedCosineSimilarity(self, movie1, movie2):
        averages = {}
        for (key, ratings) in self.data.items():
            averages[key] = (float(sum(ratings.values())) / len(ratings.values()))

            numerator = 0
            denominator1 = 0
            denominator2 = 0
        for (user, ratings) in self.data.items():
            if movie1 in ratings and movie2 in ratings:
                avg = averages[user]
                numerator += (ratings[movie1] - avg) * (ratings[movie2] - avg)
                denominator1 += (ratings[movie1] - avg) ** 2
                denominator2 += (ratings[movie2] - avg) ** 2
                
        if  (sqrt(denominator1)* sqrt(denominator2)) !=0:
            return numerator / (sqrt(denominator1)* sqrt(denominator2))
        else:
            return 0
    
    
    def normalized(self, input, userRating, min =1, max =5):
        normalizedRatings = {}
        for movie in userRating[input]:
            normal1 = 2 * (userRating[input][movie] - min) - (max-min)
            normal2 = max - min
            normal = normal1 / normal2
            normalizedRatings[movie] = normal
        return normalizedRatings
    
    def denormalized(self,input, min=1, max=5):
        denormalizedRatings = {}
        for movie in input:
            normal = (((input[movie] + 1) * (max - min)) / 2) + min
            denormalizedRatings[movie] = normal
        return denormalizedRatings

    def adjustedCosineSimilarityRecommender(self, user):
        rateForBandThatUserNotRated = {}
        # Normalize User's ratings
        user_normalized_ratings = self.normalized(user, self.data)

        # Calculate the similarity between each Band that user not rated and each of the bands User has rated
        bands_user_rated = self.data[user].keys()

        similarities = {}


        allBands = []
        for user in self.data.keys():
            for band in self.data[user].keys():
                if band not in allBands:
                    allBands.append(band)

        # band1: the bands that user did not rate
        bandsNotRated = allBands - bands_user_rated


        for bandNotRated in bandsNotRated:
            similarities[bandNotRated] = {}
            for band_user_rated in bands_user_rated:
                similarities[bandNotRated][band_user_rated] = self.adjustedCosineSimilarity(bandNotRated, band_user_rated)
                
            
            # Predict User's normalized rating for each band that not rated
            numerator = 0
            denominator = 0

            for band, similarity in similarities[bandNotRated].items():
                numerator += similarity * user_normalized_ratings[band]
                denominator += abs(similarity)

            # Handle division by zero in case no similar items are found
            if denominator != 0:
                predicted_normalized_rating = numerator / denominator
            else:
                predicted_normalized_rating = 0  # default to zero if no valid similarities

            # Denormalize the predicted rating
            predicted_normalized_dict = {bandNotRated: predicted_normalized_rating}
            predicted_denormalized_rating = self.denormalized(predicted_normalized_dict)[bandNotRated]

            rateForBandThatUserNotRated[bandNotRated] = predicted_denormalized_rating


        recommendations = list(rateForBandThatUserNotRated.items())
        #finally sort and return
        recommendations.sort(key=lambda artistTuple: artistTuple[1], reverse = True)       
    
        return recommendations 

In [25]:
r = Recommender(data ='')

In [26]:
r.loadMovieDB('D:\\Books being read\\A Prorammers Gui to Data Mining\\A-Prorammers-Gui-to-Data-Mining\\Chapter3\\ml-100k\\')

In [ ]:
r.data

In [29]:
r.computeDeviations()

In [30]:
g = r.data['944']

In [31]:
r.slopeOneRecommendations(g)[:5]

[('1533', 5.0),
 ('1656', 4.75),
 ('1463', 4.555555555555555),
 ('1599', 4.5),
 ('1467', 4.375)]

In [28]:
r.adjustedCosineSimilarityRecommender('944')[:5]

[('1533', 5.0),
 ('1332', 5.0),
 ('1371', 4.5),
 ('1674', 4.166666666666667),
 ('1272', 4.091992350037316)]